<a href="https://colab.research.google.com/github/pendyalasharanya/Powerpuffgirls/blob/main/Campus_Info_Chatbot_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q requests beautifulsoup4 pypdf langchain-community langchain-text-splitters sentence-transformers faiss-cpu gradio groq googlesearch-python

In [121]:
import os
import re
import requests
import tempfile
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from googlesearch import search

import gradio as gr
from google.colab import userdata
from groq import Groq
from pypdf import PdfReader

from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [122]:
from google.colab import userdata
from groq import Groq

client = Groq(
    api_key=userdata.get("groq_api_key")
)

print("Groq API ready")

Groq API ready


In [123]:
COLLEGE_NAME = input("Enter college name: ")

print("College selected:", COLLEGE_NAME)

Enter college name: cbit
College selected: cbit


In [120]:
COLLEGE_DIRECTORY = {
    "gnits": "https://gnits.ac.in/",
    "g narayanamma": "https://gnits.ac.in/",
    "g narayanamma institute of technology and science": "https://gnits.ac.in/",
    "vnr": "https://vnrvjiet.ac.in/",
    "vnr vjiet": "https://vnrvjiet.ac.in/",
    "cbit": "https://www.cbit.ac.in/",
    "mgit": "https://mgit.ac.in/",
    "griet": "https://www.griet.ac.in/",
    "jntuh": "https://jntuh.ac.in/",
    "anurag": "https://anurag.edu.in/",
    "mvsr": "https://mvsrec.edu.in/"
}

def find_college_website(college_name):
    name = college_name.lower().strip()

    for key, website in COLLEGE_DIRECTORY.items():
        if key in name or name in key:
            return website

    try:
        query = college_name + " official website"
        results = list(search(query, num_results=5))

        for result in results:
            if (
                "facebook" not in result
                and "linkedin" not in result
                and "wikipedia" not in result
                and "shiksha" not in result
                and "careers360" not in result
            ):
                return result

        return results[0] if results else None

    except:
        return None


COLLEGE_WEBSITE = find_college_website(COLLEGE_NAME)

if COLLEGE_WEBSITE is None:
    raise Exception("Could not find official website. Try full college name or add it to COLLEGE_DIRECTORY.")

print("Official website found:", COLLEGE_WEBSITE)

Official website found: https://gnits.ac.in/


In [124]:
DOMAIN = urlparse(COLLEGE_WEBSITE).netloc.replace("www.", "")

KEYWORDS = [
    "about", "department", "departments", "faculty", "staff", "profile",
    "hod", "program", "programs", "course", "courses", "syllabus",
    "admission", "admissions", "academic", "calendar", "timetable",
    "time-table", "placement", "placements", "training", "career",
    "library", "hostel", "facility", "facilities", "infrastructure",
    "contact", "event", "events", "club", "clubs", "research",
    "committee", "advisory", "vision", "mission", "student",
    "exam", "notification", "circular", "transport", "sports"
]

IGNORE_EXTENSIONS = [
    ".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg",
    ".mp4", ".mp3", ".zip", ".rar"
]

MAX_PAGES = 120
TIMEOUT = 10

print("Domain:", DOMAIN)

Domain: gnits.ac.in


In [125]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_valid_url(url):
    parsed = urlparse(url)

    if DOMAIN not in parsed.netloc.replace("www.", ""):
        return False

    lower_url = url.lower()

    for ext in IGNORE_EXTENSIONS:
        if lower_url.endswith(ext):
            return False

    return True


def is_useful_url(url):
    lower_url = url.lower()

    if url.rstrip("/") == COLLEGE_WEBSITE.rstrip("/"):
        return True

    return any(keyword in lower_url for keyword in KEYWORDS)


def detect_category(url, text):
    combined = (url + " " + text[:700]).lower()

    if "faculty" in combined or "staff" in combined:
        return "Faculty / Staff"
    elif "department" in combined or "hod" in combined:
        return "Department"
    elif "course" in combined or "program" in combined or "syllabus" in combined:
        return "Courses / Programs"
    elif "placement" in combined or "training" in combined:
        return "Placements"
    elif "calendar" in combined or "timetable" in combined:
        return "Academic Calendar / Timetable"
    elif "facility" in combined or "infrastructure" in combined:
        return "Facilities"
    elif "library" in combined:
        return "Library"
    elif "hostel" in combined:
        return "Hostel"
    elif "contact" in combined:
        return "Contact"
    elif "event" in combined or "club" in combined:
        return "Events / Clubs"
    elif "admission" in combined:
        return "Admissions"
    else:
        return "General"

In [126]:
visited = set()
to_visit = [COLLEGE_WEBSITE]
documents = []

while to_visit and len(visited) < MAX_PAGES:

    url = to_visit.pop(0)

    if url in visited:
        continue

    try:
        response = requests.get(url, timeout=TIMEOUT)
        content_type = response.headers.get("Content-Type", "")

        if response.status_code != 200:
            continue

        visited.add(url)

        if "application/pdf" in content_type or url.lower().endswith(".pdf"):
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.title.string if soup.title else url

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = clean_text(soup.get_text(separator=" "))

        if len(text) > 200:
            category = detect_category(url, text)

            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": url,
                        "title": title,
                        "category": category,
                        "type": "webpage"
                    }
                )
            )

            print("Scraped:", category, "|", url)

        for link in soup.find_all("a", href=True):
            full_url = urljoin(url, link["href"]).split("#")[0]

            if is_valid_url(full_url) and is_useful_url(full_url):
                if full_url not in visited and full_url not in to_visit:
                    to_visit.append(full_url)

    except Exception as e:
        print("Failed:", url, e)

print("Total web pages scraped:", len(documents))

Scraped: Courses / Programs | https://gnits.ac.in/
Scraped: Placements | https://gnits.ac.in/about-gnits/
Scraped: Department | https://gnits.ac.in/academics/
Scraped: Faculty / Staff | https://gnits.ac.in/research/
Scraped: Courses / Programs | https://gnits.ac.in/placements/
Scraped: Courses / Programs | https://gnits.ac.in/academics/bachelors/
Scraped: Courses / Programs | https://gnits.ac.in/academics/masters/
Scraped: Events / Clubs | https://gnits.ac.in/campus-life/student-clubs/
Scraped: Department | https://gnits.ac.in/campus-life/sports/
Scraped: Courses / Programs | https://gnits.ac.in/admissions/how-to-apply/
Scraped: Department | https://gnits.ac.in/research/research-advisory-ethics-committee/
Scraped: Faculty / Staff | https://gnits.ac.in/research/academic-research/
Scraped: Faculty / Staff | https://gnits.ac.in/research/seed-grants/
Scraped: General | https://gnits.ac.in/research/patents/
Scraped: Department | https://gnits.ac.in/research/journal-publications/
Scraped: Fa

In [127]:
pdf_urls = set()

for doc in documents:
    source_url = doc.metadata["source"]

    try:
        response = requests.get(source_url, timeout=TIMEOUT)
        soup = BeautifulSoup(response.text, "html.parser")

        for link in soup.find_all("a", href=True):
            full_url = urljoin(source_url, link["href"]).split("#")[0]

            if full_url.lower().endswith(".pdf") and is_valid_url(full_url):
                pdf_urls.add(full_url)

    except:
        pass

print("PDF links found:", len(pdf_urls))

for pdf in list(pdf_urls)[:20]:
    print(pdf)

PDF links found: 860
https://documents.gnits.ac.in/wp-content/uploads/2023/08/circular_drone_08-05-2023.pdf
https://documents.gnits.ac.in/wp-content/uploads/2023/08/report_reactjs_05-07-2021.pdf
https://gnits.ac.in//wp-content/uploads/2023/09/3.4.4-2022-2023-PROOFS-_part2.pdf
https://documents.gnits.ac.in/wp-content/uploads/2023/08/circular_virtualreality_11-04-2022.pdf
https://documents.gnits.ac.in//wp-content/uploads/2023/09/Smart-Baby-Leela.pdf
https://documents.gnits.ac.in//wp-content/uploads/2023/09/CSE_Student_analysis.pdf
https://documents.gnits.ac.in//wp-content/uploads/2023/04/report_yuva.pdf
https://gnits.ac.in/wp-content/uploads/2024/10/MAGZINE-EDITION15-2022-23.pdf
https://documents.gnits.ac.in/wp-content/uploads/2022/02/y-j-sudha-rani.pdf
https://documents.gnits.ac.in/wp-content/uploads/2024/01/Utility-Essentials_23-24_Circular.pdf
https://documents.gnits.ac.in/wp-content/uploads/2022/02/papanna.pdf
https://documents.gnits.ac.in/wp-content/uploads/2024/01/G20-Cyber-securit

In [128]:
pdf_count = 0

for pdf_url in list(pdf_urls)[:35]:

    try:
        response = requests.get(pdf_url, timeout=TIMEOUT)

        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_pdf:
            temp_pdf.write(response.content)
            temp_pdf_path = temp_pdf.name

        reader = PdfReader(temp_pdf_path)
        pdf_text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                pdf_text += page_text + "\n"

        pdf_text = clean_text(pdf_text)

        if len(pdf_text) > 200:
            documents.append(
                Document(
                    page_content=pdf_text,
                    metadata={
                        "source": pdf_url,
                        "title": pdf_url.split("/")[-1],
                        "category": detect_category(pdf_url, pdf_text),
                        "type": "pdf"
                    }
                )
            )

            pdf_count += 1
            print("PDF scraped:", pdf_url)

        os.remove(temp_pdf_path)

    except Exception as e:
        print("PDF failed:", pdf_url, e)

print("Total PDFs scraped:", pdf_count)
print("Total documents:", len(documents))

PDF failed: https://gnits.ac.in//wp-content/uploads/2023/09/3.4.4-2022-2023-PROOFS-_part2.pdf Stream has ended unexpectedly


PDF scraped: https://documents.gnits.ac.in//wp-content/uploads/2023/09/Smart-Baby-Leela.pdf
PDF scraped: https://documents.gnits.ac.in//wp-content/uploads/2023/04/report_yuva.pdf
PDF scraped: https://gnits.ac.in/wp-content/uploads/2024/10/MAGZINE-EDITION15-2022-23.pdf
PDF scraped: https://documents.gnits.ac.in/wp-content/uploads/2024/01/Utility-Essentials_23-24_Circular.pdf
PDF scraped: https://documents.gnits.ac.in/wp-content/uploads/2024/01/G20-Cyber-security-awarenss_23-24_circular.pdf
PDF scraped: https://documents.gnits.ac.in//wp-content/uploads/2023/09/sravani-1.pdf
PDF scraped: https://documents.gnits.ac.in//wp-content/uploads/2023/09/125.pdf
PDF scraped: https://gnits.ac.in/wp-content/uploads/2023/06/20.pdf


PDF failed: https://gnits.ac.in/wp-content/uploads/2023/07/20230627173138.pdf Stream has ended unexpectedly


PDF failed: https://gnits.ac.in/wp-content/uploads/2023/07/20230627172643.pdf Stream has ended unexpectedly
PDF scraped: https://documents.gnits.ac.in//wp-content/uploads/2023/12/Journal_Mamatha-KLU.pdf
PDF scraped: https://documents.gnits.ac.in/wp-content/uploads/2022/02/dr-madhavi_scholars.pdf
PDF scraped: https://documents.gnits.ac.in//wp-content/uploads/2023/09/hednext-mou.pdf
PDF scraped: https://gnits.ac.in/wp-content/uploads/2023/09/BoS-Meeting.pdf


PDF failed: https://www.gnits.ac.in/wp-content/uploads/2024/01/DVL_NEP.pdf Stream has ended unexpectedly
PDF scraped: https://gnits.ac.in/wp-content/uploads/2022/04/5-food.pdf
PDF scraped: https://documents.gnits.ac.in/wp-content/uploads/2024/01/G20-Cyber-security-awarenss_23-24_Report.pdf
Total PDFs scraped: 14
Total documents: 93


In [129]:
track_a_manual_info = f"""
The exact information is not available.
"""

documents.append(
    Document(
        page_content=track_a_manual_info,
        metadata={
            "source": "Track A Project Manual Knowledge",
            "title": "Track A Campus Assistant Requirements",
            "category": "Project Information",
            "type": "manual"
        }
    )
)

print("Manual Track A knowledge added")

Manual Track A knowledge added


In [130]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 1271


In [131]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Embedding model loaded


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [132]:
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS vector database created successfully")

FAISS vector database created successfully


In [158]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [159]:
college_cache = {}

COLLEGE_DIRECTORY = {
    "gnits": "https://gnits.ac.in/",
    "g narayanamma": "https://gnits.ac.in/",
    "cbit": "https://www.cbit.ac.in/",
    "vnr": "https://vnrvjiet.ac.in/",
    "vnr vjiet": "https://vnrvjiet.ac.in/",
    "vnrvjiet": "https://vnrvjiet.ac.in/",
    "mgit": "https://mgit.ac.in/",
    "griet": "https://www.griet.ac.in/",
    "jntuh": "https://jntuh.ac.in/",
    "anurag": "https://anurag.edu.in/"
}

def find_college_website_fast(college_name):
    name = college_name.lower().strip()

    for key, website in COLLEGE_DIRECTORY.items():
        if key in name or name in key:
            return website

    return None


def build_college_database(college_name):

    key = college_name.lower().strip()

    if key in college_cache:
        return college_cache[key]

    college_website = find_college_website_fast(college_name)

    if college_website is None:
        return None, None

    if not college_website.endswith("/"):
        college_website += "/"

    domain = urlparse(college_website).netloc.replace("www.", "")

    seed_urls = [
        college_website,
        urljoin(college_website, "about/"),
        urljoin(college_website, "academics/"),
        urljoin(college_website, "departments/"),
        urljoin(college_website, "department/"),
        urljoin(college_website, "programs/"),
        urljoin(college_website, "courses/"),
        urljoin(college_website, "admissions/"),
        urljoin(college_website, "placements/"),
        urljoin(college_website, "facilities/"),
        urljoin(college_website, "contact-us/"),
    ]

    visited = set()
    to_visit = seed_urls
    docs_list = []

    MAX_FAST_PAGES = 5

    while to_visit and len(visited) < MAX_FAST_PAGES:

        url = to_visit.pop(0)

        if url in visited:
            continue

        try:
            response = requests.get(url, timeout=5)

            if response.status_code != 200:
                continue

            visited.add(url)

            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup(["script", "style", "nav", "footer", "header"]):
                tag.decompose()

            text = clean_text(soup.get_text(separator=" "))

            if len(text) > 200:
                docs_list.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": url,
                            "category": detect_category(url, text)
                        }
                    )
                )

            for link in soup.find_all("a", href=True):
                full_url = urljoin(url, link["href"]).split("#")[0]
                parsed = urlparse(full_url)

                useful_words = [
                    "department", "faculty", "staff", "program",
                    "course", "academics", "placement", "facilities",
                    "admission", "contact", "syllabus"
                ]

                if domain in parsed.netloc.replace("www.", ""):
                    if any(word in full_url.lower() for word in useful_words):
                        if full_url not in visited and full_url not in to_visit:
                            to_visit.append(full_url)

        except:
            pass

    if len(docs_list) == 0:
        return None, college_website

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(docs_list)

    db = FAISS.from_documents(chunks, embedding_model)

    college_cache[key] = (db, college_website)

    return db, college_website


def chatbot(college_name, question):

    if college_name.strip() == "":
        return "Please enter college name."

    if question.strip() == "":
        return "Please enter your question."

    db, website = build_college_database(college_name)

    if db is None:
        return "College not supported yet. Try: GNITS, CBIT, VNR VJIET, MGIT, GRIET, JNTUH, Anurag."

    q = question.lower()

    if "department" in q or "departments" in q:
        search_query = "departments academic departments engineering branches programs"
        k = 12
    elif "course" in q or "courses" in q or "program" in q:
        search_query = "courses programs undergraduate postgraduate branches departments syllabus"
        k = 12
    elif "faculty" in q or "staff" in q or "hod" in q:
        search_query = "faculty staff hod department profile"
        k = 12
    else:
        search_query = question
        k = 8

    docs = db.similarity_search(search_query, k=k)

    context = ""
    sources = []

    for doc in docs:
        context += doc.page_content + "\n\n"
        sources.append(doc.metadata["source"])

    prompt = f"""
You are a Campus Information Chatbot.

College: {college_name}
Website: {website}

Answer using only the context below.

Rules:
- Give direct and clean answer.
- If departments are asked, list department names clearly.
- If courses are asked, list course/program names clearly.
- If faculty/staff are asked, list names and roles if available.
- Do not invent information.
- If exact answer is not available, say exact information was not found.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )

    answer = response.choices[0].message.content.strip()

    answer += "\n\nSources:\n"

    for source in sorted(set(sources)):
        answer += source + "\n"

    return answer

In [160]:
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #eef2ff, #f8fafc);
}

#title {
    text-align: center;
    color: #1e3a8a;
    font-size: 34px;
    font-weight: bold;
}

#subtitle {
    text-align: center;
    color: #475569;
    font-size: 16px;
    margin-bottom: 20px;
}

.input-box textarea {
    border-radius: 12px !important;
}

.output-box textarea {
    border-radius: 12px !important;
}

button {
    border-radius: 12px !important;
    font-weight: bold !important;
}
"""

with gr.Blocks(css=custom_css) as app:

    gr.Markdown(
        """
        <div id="title">Campus Info Chatbot AI Agent</div>
        <div id="subtitle">Enter college name and ask campus-related questions</div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            college_input = gr.Textbox(
                label="College Name",
                placeholder="Example: GNITS, CBIT, VNR VJIET, MGIT",
                elem_classes="input-box"
            )

            question_input = gr.Textbox(
                label="Your Question",
                placeholder="Example: Which departments are available?",
                lines=3,
                elem_classes="input-box"
            )

            submit_btn = gr.Button("Get Answer")

        with gr.Column(scale=1):
            answer_output = gr.Textbox(
                label="Answer",
                lines=15,
                elem_classes="output-box"
            )

    submit_btn.click(
        fn=chatbot,
        inputs=[college_input, question_input],
        outputs=answer_output
    )

app.launch(share=True, debug=False)

/tmp/ipykernel_15285/3394596799.py:34: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a6bc90ad923e7a714.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [144]:
vector_db.save_local("campus_faiss_db")

print("Saved FAISS database as campus_faiss_db")

Saved FAISS database as campus_faiss_db
